## Time Dependent vs. Steady State Temperature Comparison

We've now built up the tools to model the pressures, velocities, and temperatures of subduction zones for both steady state and time-dependent problems. Given that the steady-state assumption allows for a much less computationally expensive problem, it is useful to check if it is an accurate approximation of a real world time-dependent problem.

Because we are interested in eventually using these models to predict the hydration state of the subducting slabs, we will solve time-dependent problems of increasing duration, and examine the convergance of temperatures in the slab of the benchmark SZ problem towards the steady state solution:

Set file path information:

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

Import relevant classes and functions from previous subduction zone notebooks:

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry

from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem

Import other required modulues and set output folder

In [ ]:
import fenics_sz.utils
import numpy as np
import matplotlib.pyplot as pl
import copy
import pyvista as pv
import pathlib
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

#### Set up and solve a dislocation creep problem with a benchmark geometry:

Setting resolution for FEM: (we'll keep this consistent for both problems)

In [ ]:
resscale = 5.0

Benchmark parameters: (dislocation creep)

In [ ]:
xs = [0.0, 140.0, 240.0, 400.0]
ys = [0.0, -70.0, -120.0, -200.0]
lc_depth = 40
uc_depth = 15
coast_distance = 0
extra_width = 0
sztype = 'continental'
io_depth = 154.0

A = 100.0      # age of subducting slab (Myr)
qs = 0.065      # surface heat flux (W/m^2)
Vs = 100.0      # slab speed (mm/yr)

Create the slab, subduction zone geometry, and the steady state problem:

In [ ]:
slab = create_slab(xs, ys, resscale, lc_depth)
geom_case_2 = create_sz_geometry(slab, resscale, sztype, io_depth, extra_width, 
                                        coast_distance, lc_depth, uc_depth)
sz_case_2_steady = SteadyDislSubductionProblem(geom_case_2, A=A, Vs=Vs, sztype=sztype, qs=qs)

Solve the steady state problem

In [ ]:
print("Solving steady state flow with dislocation creep rheology...")
sz_case_2_steady.solve()

We can plot the results to confirm everything looks as expected:

In [ ]:
plotter_dis_steady = fenics_sz.utils.plot.plot_scalar(sz_case_2_steady.T_i, scale=sz_case_2_steady.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
fenics_sz.utils.plot.plot_vector_glyphs(sz_case_2_steady.vw_i, plotter=plotter_dis_steady, factor=0.1, gather=True, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz_case_2_steady.v0))
fenics_sz.utils.plot.plot_vector_glyphs(sz_case_2_steady.vs_i, plotter=plotter_dis_steady, factor=0.1, gather=True, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz_case_2_steady.v0))
sz_case_2_steady.geom.pyvistaplot(plotter=plotter_dis_steady, color='green', width=2)
cdpt = sz_case_2_steady.geom.slab_spline.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter_dis_steady, render_points_as_spheres=True, point_size=10.0, color='green')
fenics_sz.utils.plot.plot_show(plotter_dis_steady)

And save the results if we like

In [ ]:
fenics_sz.utils.plot.plot_save(plotter_dis_steady, output_folder / "sz_problem_case2_steady_solution.png")

#### Time-dependent problem

Setting up time-dependent problem using same geometry:

In [ ]:
sz_case_2_td = TDDislSubductionProblem(geom_case_2, A=A, Vs=Vs, sztype=sztype, qs=qs)

Define a few additional paramaters needed to solve the time-dependent problem

In [ ]:
tf = 25
dt = 0.05
theta = 0.5
rtol = 1.e-3

Now we call our time-dependent solve function

In [ ]:
sz_case_2_td.solve(tf, dt, theta=theta, rtol=rtol)

And again, plotting as before to confirm that everything has run properly:

In [ ]:
plotter_dis_td = fenics_sz.utils.plot.plot_scalar(sz_case_2_td.T_i, scale=sz_case_2_td.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
fenics_sz.utils.plot.plot_vector_glyphs(sz_case_2_td.vw_i, plotter=plotter_dis_td, factor=0.1, gather=True, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz_case_2_td.v0))
fenics_sz.utils.plot.plot_vector_glyphs(sz_case_2_td.vs_i, plotter=plotter_dis_td, factor=0.1, gather=True, color='k', scale=fenics_sz.utils.mps_to_mmpyr(sz_case_2_td.v0))
sz_case_2_td.geom.pyvistaplot(plotter=plotter_dis_td, color='green', width=2)
cdpt = sz_case_2_td.geom.slab_spline.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter_dis_td, render_points_as_spheres=True, point_size=10.0, color='green')
fenics_sz.utils.plot.plot_show(plotter_dis_td)
# fenics_sz.utils.plot.plot_save(plotter_dis_td, output_folder / "sz_problem_case2td_solution.png")

In [ ]:
fenics_sz.utils.plot.plot_save(plotter_dis_td, output_folder / "sz_problem_case2td_solution.png")

#### Plotting the differences in temperature

Now let's plot the differences between the two plots we've generated.

We'll be returning a data structure that will store a grid of all of our temperature differences between the two plots we want to compare. In order to scale the color bars such that a temp. difference of 0 is in the center of the color gradient, we'll define a funciton to find the maximum magnitude within our collection of temperature difference, and use that to scale the color bar.

In [ ]:
def get_max_magnitude(range):
    range_abs = np.abs(range)
    max_magnitude=range_abs[np.argmax(range_abs)]
    return max_magnitude

Now we'll define the function that will let us plot temperature differences overlaid on the benchmark subduction geometry

In [ ]:
def plot_temp_diff(case_1, case_2, show_coupling_point=True, filename=None, **pv_kwargs):

    """
    Use pyvista to find and plot the temperature difference of two subduction zone problems

    Arguments:
        * case_1 - solved sz problem: base case
        * case_2 - solved sz problem to compare against case_1

    Keyword Arguments:
        * show_coupling_point  - show coupling point when plotting (defaults to True)
        * filename             - name of file when exporting (defauls to None: no export)

    """
    dis_td_grid = fenics_sz.utils.plot.grids_scalar(case_1.T_i)[0]
    dis_steady_grid = fenics_sz.utils.plot.grids_scalar(case_2.T_i)[0]

    diffgrid_2=fenics_sz.utils.plot.pv_diff(dis_td_grid, dis_steady_grid)

    plotter_diff = pv.Plotter()

    #Setting bounds for temp scalar bar:
    range = diffgrid_2.get_data_range("T")
    max_abs_value = get_max_magnitude(range)
    clim = [-max_abs_value, max_abs_value]

    plotter_diff.add_mesh(diffgrid_2, cmap='coolwarm', clim=clim, scalar_bar_args={'vertical':True, 'position_y':0.3})

    geom_case_2.pyvistaplot(plotter=plotter_diff, color='green', width=2)

    if show_coupling_point==True:
        cdpt = slab.findpoint('Slab::FullCouplingDepth')
        fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter_diff, render_points_as_spheres=True, point_size=10.0, color='red')
   
    plotter_diff.show_bounds(show_zlabels=True, bold=False, font_size=8, use_3d_text=False, ytitle="km", xtitle="km", n_xlabels=4, n_ylabels=4)

    plotter_diff.enable_parallel_projection()
    plotter_diff.view_xy()
    plotter_diff.show()

    if filename != None:
        fenics_sz.utils.plot.plot_save(plotter_diff, filename)
        print("File Exported")

In [ ]:
plot_temp_diff(sz_case_2_td, sz_case_2_steady, show_coupling_point=True, filename=output_folder / "benchmark_steady_vs_td_test.png")

#### Plotting geotherms to check for steady-state convergence

If we want to get the results for the stokes and heat equations for multiple durations of a time-dependent problem, we need to run our time-dependent solve function for each of those durations. Our plotting function below takes an array of solved SZ problems as an input, so time-dependent problems of multiple durations can be solved and then passed into the plotter.

In [ ]:
def plot_diff_solutions(sz_probs, probe, steady_state=None, legend=None, filename=None):

    """
    Arguements:
        * sz_probs - an array of solved subduction zones problems to probe the temperature of
        * probe    - 'slab' or 'moho' : the part of the subduction zone geometry along which to probe the temperature

    Keyword Arguments:
        * steady_state  - a solved steady-state sz problem
        * legend        - an array of labels for each of the geotherms that are plotted
        * filename      - string: name of file when exporting (defauls to None: no export)

    """

    #make sure user is selecting a path to probe that's supported by this function
    assert probe in ['slab', 'moho']

    fig = pl.figure()
    ax = fig.gca()


    i=0
    if probe =='slab':
        for sz in sz_probs:
            slabpoints = np.array([[curve.points[0].x, curve.points[0].y, 0.0] for curve in sz.geom.slab_spline.interpcurves])
            cinds, cells = fenics_sz.utils.mesh.get_cell_collisions(slabpoints, sz.mesh)
            ax.plot(sz.T_i.eval(slabpoints, cells)[:,0], -slabpoints[:,1], label=str((legend[i], 'MYr')))
            i+=1
        
        if steady_state != None: #plot the steady-state temp. curve using a dotted line
            slabpoints = np.array([[curve.points[0].x, curve.points[0].y, 0.0] for curve in steady_state.geom.slab_spline.interpcurves])
            cinds, cells = fenics_sz.utils.mesh.get_cell_collisions(slabpoints, steady_state.mesh)
            ax.plot(steady_state.T_i.eval(slabpoints, cells)[:,0], -slabpoints[:,1], label='Steady State', linestyle='dashed')
            
        ax.set_title('Slab surface temperatures')


    elif probe == 'moho':
        for sz in sz_probs:
            slabmoho = copy.deepcopy(sz.geom.slab_spline)
            slabmoho.translatenormalandcrop(-7.0) # get a path 7km normal to the slab surface, cropped such that it fits w/n the problem's geometry
            slabmohopoints = np.array([[curve.points[0].x, curve.points[0].y, 0.0] for curve in slabmoho.interpcurves])
            mcinds, mcells = fenics_sz.utils.mesh.get_cell_collisions(slabmohopoints, sz.mesh)
            ax.plot(sz.T_i.eval(slabmohopoints, mcells)[:,0], -slabmohopoints[:,1], label=str((legend[i], 'MYr')))
            i+=1
        
        if steady_state != None:
            slabmoho = copy.deepcopy(steady_state.geom.slab_spline)
            slabmoho.translatenormalandcrop(-7.0)
            slabmohopoints = np.array([[curve.points[0].x, curve.points[0].y, 0.0] for curve in slabmoho.interpcurves])
            mcinds, mcells = fenics_sz.utils.mesh.get_cell_collisions(slabmohopoints, steady_state.mesh)
            ax.plot(steady_state.T_i.eval(slabmohopoints, mcells)[:,0], -slabmohopoints[:,1], label='Steady State', linestyle='dashed')

        ax.set_title('Moho temperatures')

    ax.set_xlabel('T ($^\circ$C)')
    ax.set_ylabel('z (km)')
    ax.legend()
    ax.invert_yaxis()

    if filename != None:
        fig.savefig(output_folder / filename)
        print(filename, "exported")

Setting the durations of time for which we wish to solve the time-dependent problem for:

In [ ]:
time_lengths = [1,2,5,10,25]
sz_problems = []

In [ ]:
#Setup params
A = 100.0      # age of subducting slab (Myr)
qs = 0.065      # surface heat flux (W/m^2)
Vs = 100.0      # slab speed (mm/yr)

#Solver params
dt = 0.05
theta = 0.5
rtol = 1.e-3

Solve the time-dependent problem for each desired duration and append each solution to our array of sz problem solutions:

In [ ]:
sz_problems = []

for tf in time_lengths:
    sz_case_2_td = TDDislSubductionProblem(geom_case_2, A=A, Vs=Vs, sztype=sztype, qs=qs)
    sz_case_2_td.solve(tf, dt, theta=theta, rtol=rtol)
    sz_problems.append(sz_case_2_td)
    print(tf, " MYr case solved")

Now call the plotting function, passing in our array of solved sz problems as an input

In [ ]:
plot_diff_solutions(sz_problems, probe='slab', steady_state=sz_case_2_steady, legend=time_lengths, filename="slab_temps.png")
plot_diff_solutions(sz_problems, probe='moho', steady_state=sz_case_2_steady, legend=time_lengths, filename="moho_temps.png")

We can see how even after as little as 5 MYr into the run, most of the time-dependent simulation of the slab temperature has converged fairly well towards the steady-state solution, even more so by 10 MYr.